# Z3-C#-16c — Meal-Planner : capstone patient (restrictions nutritionnelles)

**Navigation** : [Index](README.md) | [Index SMT](../README.md) | [Index SymbolicAI](../../README.md) | [<< 16b - couche de données](Z3-16b-Meal-Planner-Data-External-Python.ipynb)

Port C# natif (Microsoft.Z3) du grain [16c Python](Z3-16c-Meal-Planner-Patient-Capstone-Python.ipynb) — même corpus, même encodage, même trajectoire pédagogique, API C#.

*Compagnon « capstone » de la jambe [16](Z3-16-Meal-Planner-Python.ipynb) (modélisation jouet) → [16b](Z3-16b-Meal-Planner-Data-External-Python.ipynb) (couche de données réelles) → **16c (ce notebook) : le patient** → [16d](Z3-16d-Meal-Planner-Convergence-Scale-Python.ipynb) (convergence à l'échelle).*

Le capstone : un **patient** impose des **restrictions nutritionnelles** (énergie bornée, protéines
minimum, lipides maximum) et le solveur doit composer un menu multi-jours qui les satisfait.
Le carnet Python est un **port fidèle du C# `08_Meal_Planner_Patient_Capstone`**
([Z3-Linq2Z3](../Z3-Linq2Z3/08_Meal_Planner_Patient_Capstone.ipynb)) ; ce carnet-ci **referme la
boucle** : il rejoue le même capstone avec le binding natif `Microsoft.Z3`, sans la couche `Z3.Linq`.

> **Ce que ce port C# apporte.** Le C# `08` d'origine (Z3.Linq2Z3) posait
> `DefaultCollectionHandling = CollectionHandling.Array` pour traduire un `int[][]` en *SMT Array* —
> une couche de traduction qui masque l'encodage réellement envoyé au solveur. Ici, pas de Z3.Linq :
> on déclare les variables `Int` une à une, comme en z3-py, et on écrit **l'idiome index +
> disjonction** à la main. Le prix d'entrée (30 variables, 90 disjonctions) est exactement ce que la
> couche de binding vous épargnait — et c'est ce que ce carnet rend visible.

```
Patient ──> Menu (1 jour) ──> Plat (1 créneau) ──> Denrées (nutrition Ciqual)
            [energie?]       [entrée | plat | accomp.]
```

**Deux mouvements** : (1) un **squelette structural** à grande échelle (semaine 7×5, sans
nutrition) ; (2) le **capstone patient** curé (nutrition contrainte).

## 0. Dépendances : le cache produit par 16b

Ce notebook **consomme** le cache solveur-usable régénéré par
[16b](Z3-16b-Meal-Planner-Data-External-Python.ipynb) (`data/meals/mealplan_cache.json`). Le
chargement est **natif** (`System.Text.Json`) : aucune dépendance Python. Pas de solveur ici tant
que le cache est absent — échec **bruyant** (règle SOTA : pas de corpus de substitution).

In [1]:
#r "nuget: Microsoft.Z3"
#load "Z3NativeLoader.cs"
using Microsoft.Z3;
using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.IO;
using System.Linq;
using System.Text.Json;

// Bibliotheque native de Z3 hors Windows et macOS Intel (voir Z3NativeLoader.cs)
Z3NativeLoader.Register(typeof(Context).Assembly);

// Contexte Z3 partage par toutes les cellules (meme moteur C++ que z3-solver Python).
var ctx = new Context();
Console.WriteLine($"Moteur : Microsoft.Z3 {Microsoft.Z3.Version.FullVersion}");

// --- Ancrage CWD-independant : le corpus vit dans Z3-API/data/meals/ (mirror du pin C# #8901).
string MealBase() {
    var cwd = new DirectoryInfo(Directory.GetCurrentDirectory());
    if (cwd.Name == "Z3-API") return Path.Combine(cwd.FullName, "data", "meals");
    for (var d = cwd; d != null; d = d.Parent) {
        var c = Path.Combine(d.FullName, "MyIA.AI.Notebooks", "SymbolicAI", "SMT", "Z3-API", "data", "meals");
        if (Directory.Exists(c)) return c;
    }
    throw new FileNotFoundException("Serie Z3-API introuvable depuis " + cwd.FullName);
}
var cachePath = Path.Combine(MealBase(), "mealplan_cache.json");
if (!File.Exists(cachePath)) {
    Console.WriteLine($"[CACHE ABSENT] {cachePath}");
    Console.WriteLine("Executez d'abord Z3-Python-16b-Meal-Planner-Data-External (couche de donnees).");
    throw new FileNotFoundException("Cache absent", cachePath);
}
var serieDir = Directory.GetParent(MealBase())!.Parent!.FullName;
Console.WriteLine($"Cache present : {Path.GetRelativePath(serieDir, cachePath)} "
    + $"({new FileInfo(cachePath).Length / 1024.0:F0} Ko)");

// --- Derivation du creneau (Ordre 1..5) depuis les cats RecipeML : meme table que le port Python.
int OrdreFromCats(string[] cats) {
    var j = string.Join(" ", cats).ToLowerInvariant();
    if (new[] { "appetizer", "starter", "soup" }.Any(k => j.Contains(k))) return 1;
    if (new[] { "main dish", "beef", "pork", "chicken", "meat", "fish" }.Any(k => j.Contains(k))) return 2;
    if (new[] { "vegetable", "salad", "rice", "pasta", "potato", "bread", "side" }.Any(k => j.Contains(k))) return 3;
    if (new[] { "cheese", "dairy", "yogurt", "cream" }.Any(k => j.Contains(k))) return 4;
    if (new[] { "dessert", "cake", "cookie", "pie", "fruit", "sweet", "tart" }.Any(k => j.Contains(k))) return 5;
    return 0;
}

var constituants = new List<string>();
var allPlats = new List<(string Nom, int Ordre, double[] Comp)>();
int nTotal;
var swLoad = Stopwatch.StartNew();
using (var doc = JsonDocument.Parse(File.ReadAllText(cachePath))) {
    foreach (var e in doc.RootElement.GetProperty("constituants").EnumerateArray())
        constituants.Add(e.GetString()!);
    nTotal = doc.RootElement.GetProperty("n_total").GetInt32();
    foreach (var r in doc.RootElement.GetProperty("recipes").EnumerateArray()) {
        var t = (r.GetProperty("title").GetString() ?? "").Trim();
        var o = OrdreFromCats(r.GetProperty("cats").EnumerateArray().Select(c => c.GetString()!).ToArray());
        if (o < 1) continue;                            // orphelin ecarte
        allPlats.Add((t[..Math.Min(44, t.Length)],      // tronque a 44 (miroir du port Python)
                      o,
                      r.GetProperty("vec").EnumerateArray().Select(v => v.GetDouble()).ToArray()));
    }
}
int C = constituants.Count;
swLoad.Stop();
Console.WriteLine($"Cache charge en {swLoad.Elapsed.TotalSeconds:F2}s : {allPlats.Count} recettes exploitables "
    + $"(sur {nTotal} brutes), {C} constituants.");
Console.WriteLine("Constituants : " + string.Join(" | ",
    constituants.Select((n, i) => $"[{i}] {n.Split(',')[0].Trim()}")));

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installing Packages Microsoft.Z3

Moteur : Microsoft.Z3 Z3 4.12.2.0


Cache present : data\meals\mealplan_cache.json (89 Ko)


Cache charge en 0,02s : 447 recettes exploitables (sur 1465 brutes), 5 constituants.


Constituants : [0] Energie | [1] Protéines | [2] Glucides (g/100 g) | [3] Lipides (g/100 g) | [4] Sel chlorure de sodium (g/100 g)


### Lecture du résultat — l'entonnoir du corpus

Le cache ne retient que **447 recettes exploitables sur 1465 brutes** (31 %) : tout ce qui n'a
pas de fiche nutritionnelle complète sur les cinq constituants est écarté en amont, en 16b — le
solveur ne verra jamais les manquants. Les cinq constituants suivis couvrent l'énergie, les
protéines, les glucides, les lipides et le sel : c'est la granularité maximale dont disposera la
couche de contraintes du Mouvement 2.

Le chargement tient en quelques millisecondes pour 89 Ko : la phase « données » est déjà
déconnectée de la phase « solveur », et elle ne dépend plus d'un interpréteur Python — c'est le
contrat de la couche 16b, un artefact solveur-usable régénéré à la demande, lu ici par
`System.Text.Json`.

## Mouvement 1 — squelette structural à grande échelle (7 jours × 5 créneaux)

Avant la nutrition, le **squelette** : une semaine de 7 jours × 5 créneaux, où chaque créneau
choisit un plat dans sa catégorie. C'est l'occasion d'examiner l'idiome « tableau imbriqué »
(`int[][]`) en SMT — et ce que chaque binding en fait.

> **Divergence de port (honnête, et c'est le sujet).** Le C# `08` d'origine posait
> `DefaultCollectionHandling = CollectionHandling.Array` pour traduire `int[][]` en un *SMT Array*
> (théorie des tableaux de McCarthy, `Select`/`Store`). z3-py n'impose pas cette couche : on
> déclare une `list[list[Int]]` — chaque cellule est une variable `Int` indépendante. Ce carnet C#
> natif fait **le même choix que z3-py** : une grille plate de variables `Int`, sans couche de
> traduction. Fonctionnellement équivalent pour ce squelette ; la vraie théorie des Array est
> l'objet du notebook [15](Z3-15-Nested-Arrays-2D-Python.ipynb).

### 1.1 Corpus + créneaux : pourquoi trier par Ordre

In [2]:
// Semaine 7x5 : on groupe les plats par creneau, ranges contigus [lo,hi] par categorie.
const int JOURS = 7, CRENEAUX = 5;
// tri par (ordre, nom) pour des ranges contigus par creneau (le tri n'affecte que lo/hi, pas le modele M2).
// StringComparer.Ordinal = l'ordre de code point de Python, pas la collation culturelle de .NET.
var plats = allPlats.OrderBy(p => p.Ordre).ThenBy(p => p.Nom, StringComparer.Ordinal).ToList();
var lo = new int[CRENEAUX];
var hi = new int[CRENEAUX];
for (int o = 1; o <= CRENEAUX; o++) {
    var idxs = Enumerable.Range(0, plats.Count).Where(i => plats[i].Ordre == o).ToList();
    lo[o - 1] = idxs.Count > 0 ? idxs[0] : 0;
    hi[o - 1] = idxs.Count > 0 ? idxs[^1] : -1;
}
Console.WriteLine($"{plats.Count} plats ranges en 5 creneaux :");
for (int o = 0; o < CRENEAUX; o++) {
    int n = hi[o] < lo[o] ? 0 : hi[o] - lo[o] + 1;
    Console.WriteLine($"  creneau {o + 1} (ordre {o + 1}) : [{lo[o]}..{hi[o]}] ({n} candidats)");
}

447 plats ranges en 5 creneaux :


  creneau 1 (ordre 1) : [0..45] (46 candidats)


  creneau 2 (ordre 2) : [46..99] (54 candidats)


  creneau 3 (ordre 3) : [100..227] (128 candidats)


  creneau 4 (ordre 4) : [228..237] (10 candidats)


  creneau 5 (ordre 5) : [238..446] (209 candidats)


### Lecture du résultat — des créneaux très inégalement fournis

La somme des cinq fenêtres (46 + 54 + 128 + 10 + 209) redonne exactement les 447 plats du
cache : le tri par `Ordre` a réparti tout le corpus, sans reste. Mais la répartition est loin
d'être uniforme — le créneau 4 n'offre que **10 candidats** quand le créneau 5 en compte 209 :
vingt fois plus. C'est une donnée d'entrée pour la suite : tout ce que le modèle demandera au
créneau 4 devra passer par cette fenêtre étroite.

Le point clé est structurel : chaque créneau est maintenant un **intervalle contigu d'indices**
(`[0..45]`, `[46..99]`, ...). C'est ce qui permet au modèle d'encadrer chaque choix par deux
bornes d'entiers (`WithBounds`) plutôt que par une disjonction de dizaines de valeurs —
l'entier borné est le prix d'entrée le plus bas qu'un solveur accepte.

### 1.2 Le théorème hiérarchique + deux variantes

`WithBounds` encadre chaque créneau dans sa range contigüe. **Variante A** : montée en gamme
(chaque jour fait *mieux* que le précédent, même créneau). **Variante B** : variété totale
(`MkDistinct` croisé sur la colonne d'un même créneau, tous jours confondus).

In [3]:
// M1 : squelette 7x5 + Variante A (montee en gamme) + Variante B (Distinct cross-row).
IntExpr[][] MakePlan(string prefix) => Enumerable.Range(0, JOURS)
    .Select(j => Enumerable.Range(0, CRENEAUX)
        .Select(c => ctx.MkIntConst($"{prefix}_j{j}_c{c}")).ToArray()).ToArray();
IEnumerable<BoolExpr> WithBounds(IntExpr[][] plan) {
    for (int j = 0; j < JOURS; j++)
        for (int c = 0; c < CRENEAUX; c++) {
            yield return ctx.MkGe(plan[j][c], ctx.MkInt(lo[c]));
            yield return ctx.MkLe(plan[j][c], ctx.MkInt(hi[c]));
        }
}
long Val(Model m, IntExpr e) => ((IntNum)m.Evaluate(e, true)).Int64;
string Trunc(string s, int n) => s[..Math.Min(n, s.Length)];
void PrintPlan(IntExpr[][] plan, Model mdl, string titre) {
    Console.WriteLine(titre);
    for (int j = 0; j < JOURS; j++) {
        var row = new List<string>();
        for (int c = 0; c < CRENEAUX; c++) {
            int v = (int)Val(mdl, plan[j][c]);
            row.Add($"{Trunc(plats[v].Nom, 14),-14}");
        }
        Console.WriteLine($"  J{j + 1}: " + string.Join(" | ", row));
    }
}

var swA = Stopwatch.StartNew();
// Variante A : montee en gamme, chaque jour > precedent (meme creneau)
var sA = ctx.MkSolver();
var planA = MakePlan("a");
foreach (var c in WithBounds(planA)) sA.Assert(c);
for (int j = 0; j < JOURS - 1; j++)
    for (int c = 0; c < CRENEAUX; c++)
        sA.Assert(ctx.MkLt(planA[j][c], planA[j + 1][c]));
bool satA = sA.Check() == Status.SATISFIABLE;
Model mA = null;
if (satA) mA = sA.Model;
Console.WriteLine($"[A] Montee en gamme : {(satA ? "resolue" : "UNSAT")} en {swA.Elapsed.TotalMilliseconds:F0} ms");

// Variante B : Distinct cross-row par creneau (tous jours differents dans la meme categorie)
var swB = Stopwatch.StartNew();
var sB = ctx.MkSolver();
var planB = MakePlan("b");
foreach (var c in WithBounds(planB)) sB.Assert(c);
for (int c = 0; c < CRENEAUX; c++)
    if (hi[c] >= lo[c] && (hi[c] - lo[c] + 1) >= JOURS)
        sB.Assert(ctx.MkDistinct(Enumerable.Range(0, JOURS).Select(j => planB[j][c]).ToArray()));
bool satB = sB.Check() == Status.SATISFIABLE;
Console.WriteLine($"[B] Permutation totale (Distinct cross-row) : {(satB ? "resolue" : "UNSAT")} en {swB.Elapsed.TotalMilliseconds:F0} ms");
if (satA) PrintPlan(planA, mA, "\n== Variante A (montee en gamme) ==");

[A] Montee en gamme : resolue en 74 ms


[B] Permutation totale (Distinct cross-row) : resolue en 16 ms



== Variante A (montee en gamme) ==


  J1: 'ncapriata Di  | (Sort-Of) Swee | 'sense and Sen | 3-Step Cheesec | #1 Lemon Bars 


  J2: 4 B's Restaura | 1-Pot Pastitsi | ( From Bread M | Acapulco Rice  | #10 Cake      


  J3: 5-Minute Brocc | 1-Pot: Pastits | ( From Bread M | Almond Cheesec | $100 Chocolate


  J4: 7 Layer Dip    | 10 Minute Szec | 1-2-3 Meurbete | Almond Fried I | 'gimme Both' P


  J5: 90-Minute Soft | 16th-Street St | 1-Pot Creamy C | Almond Ice Cre | (Sort of Light


  J6: Abadoo's Walnu | 19-Alarm Chili | 1-Pot: Creamy  | Almond Poppyse | 1 Egg Butter S


  J7: Abalone Soup # | 2 Pungent Bbq  | 10-Minute Lasa | Almond, Lemon  | 1,000 Calorie-


### Lecture du résultat — deux exigences, deux encodages, un même verdict

Les deux variantes sont triviales à résoudre à cette échelle : la montée en gamme (A) et la
permutation totale (B) rendent un modèle en quelques dizaines de millisecondes chacune. L'écart
n'est pas un résultat de complexité — c'est un indice : A impose une chaîne d'inégalités entre
jours voisins, B se contente d'un `Distinct` croisé par colonne. À 7 jours, aucune des deux ne
coûte ; la question du coût ne se posera qu'à l'échelle du carnet 16d.

Sur la solution A, chaque colonne (un créneau, lu de J1 à J7) affiche des noms dans l'ordre
alphabétique croissant — et c'est la contrainte elle-même, rendue visible : le tri de 16b étant
alphabétique au sein de chaque créneau, exiger un indice croissant par colonne fait croître les
noms dans l'ordre du dictionnaire. Les noms tronqués à 14 caractères sont un choix d'affichage ;
les noms complets vivent dans le cache.

## Mouvement 2 — le capstone patient : la nutrition force la curation

Le squelette M1 ignore la nutrition. Lui ajouter des restrictions patient à l'échelle pleine
(`7×5×R×C`) ferait exploser le modèle. D'où la **curation** : on réduit à **2 menus × 3 créneaux ×
5 candidats × 3 constituants** — un sous-ensemble à taille humaine où le capstone reste lisible.

> **Note de fidélité** : le C# `08` référence dans sa prose une classe upstream `Patient {
> Restriction[] Restrictions }` (chaque restriction porte un `Min`/`Max` par constituant, `-1` =
> pas de borne), qui **n'existe pas dans ce dépôt** (fork externe `Z3.LinqBinding`). Le notebook
> `08` lui-même **inline** le patient comme quatre entiers et n'a **aucune** classe
> `Patient`/`Restriction`. Ce port C# reste **fidèle à `08`** (patient inline) plutôt que de
> reconstruire la classe upstream absente — ne pas lui attribuer une structure que `08` n'a pas.

### 2.1 Curation : 2 menus × 3 créneaux × 5 candidats × 3 constituants

In [4]:
// Curation : pool de 5 candidats par creneau 1..3 (cache order, PAS le tri M1), 3 constituants.
const int NB_MENUS = 2, NB_PLATS = 3, CAND = 5;
// 3 constituants cles : [0] Energie kJ, [1] Proteines, [3] Lipides (le 2 = Glucides est SKIPPE).
const int cEnergie = 0, cProt = 1, cLip = 3;
var CONST = new[] { cEnergie, cProt, cLip };
var NOMK = new[] { "energie(kJ)", "proteines(g)", "lipides(g)" };
int K = CONST.Length;
Console.WriteLine("Constituants suivis :");
for (int k = 0; k < K; k++)
    Console.WriteLine($"  k={k} -> cache[{CONST[k]}] {constituants[CONST[k]].Split(',')[0].Trim()}");

var pool = new List<(string Nom, int Ordre, double[] Comp)>();
for (int o = 1; o <= NB_PLATS; o++)
    pool.AddRange(allPlats.Where(p => p.Ordre == o).Take(CAND));   // cache order, fidele a 08
// Math.Round(double) = arrondi half-to-even par defaut, comme round() Python.
int Teneur(int i, int constIdx) => (int)Math.Round(pool[i].Comp[constIdx]);
int Slot(int m, int p) => m * NB_PLATS + p;
Console.WriteLine($"\nPool curee : {pool.Count} plats ({NB_PLATS} creneaux x {CAND} candidats)");
Console.WriteLine("Apercu (creneau : candidats -> energie kJ) :");
for (int p = 0; p < NB_PLATS; p++) {
    var noms = Enumerable.Range(p * CAND, CAND)
        .Select(i => $"{Trunc(pool[i].Nom.Trim(), 18)}({Teneur(i, cEnergie)})");
    Console.WriteLine($"  creneau {p + 1} : " + string.Join(", ", noms));
}

Constituants suivis :


  k=0 -> cache[0] Energie


  k=1 -> cache[1] Protéines


  k=2 -> cache[3] Lipides (g/100 g)



Pool curee : 15 plats (3 creneaux x 5 candidats)


Apercu (creneau : candidats -> energie kJ) :


  creneau 1 : 'ncapriata Di Fave(5922), 4 B's Restaurant T(3409), 5-Minute Broccoli (6415), 7 Layer Dip(11374), 90-Minute Soft Pre(7426)


  creneau 2 : (Sort-Of) Sweet an(1018), 1-Pot: Pastitsio G(15054), 1-Pot Pastitsio Go(15054), 10 Minute Szechuan(1178), 16th-Street Stew(10678)


  creneau 3 : 'sense and Sensibi(5329), ( From Bread Mix )(4421), ( From Bread Mix )(13568), 1-2-3 Meurbeteig D(14026), 1-Pot: Creamy Chic(3471)


### Lecture du résultat — quinze plats, un facteur 14 d'énergie

La curation passe de 447 à 15 plats : 5 candidats par créneau, 3 créneaux. Le pool est assez
étroit pour que l'idiome de linking reste lisible (90 disjonctions au total), assez riche pour
que la contrainte patient morde — c'est l'équilibre que le Mouvement 2 cherche.

L'aperçu énergie le montre : au sein du seul créneau 2, les candidats vont de 1018 kJ (Sort-Of
Sweet and Sour) à 15054 kJ (1-Pot Pastitsio) — un facteur 14 entre deux plats du même créneau.
Aucune heuristique de choix ne pourrait satisfaire une borne d'énergie sans explorer : c'est ce
qui justifie le solveur. On note enfin que seuls trois constituants sur cinq sont suivis — les
indices de cache 0, 1 et 3 (énergie, protéines, lipides), projetés en `k = 0, 1, 2` du modèle :
exactement ceux que le patient borne ; glucides et sel traversent le cache sans entrer dans le
modèle.

### 2.2 Trois matrices d'entiers : `PlatId`, `Comp`, `MenuNut`

| Matrice | Forme | Rôle |
|---|---|---|
| `platid[m][p]` | 2×3 | index pool du plat choisi par (menu, créneau) |
| `comp[slot][k]` | 6×3 | **composition liée** du slot aplati (variable auxiliaire) |
| `menunut[m][k]` | 2×3 | somme nutritionnelle par menu |

`comp` est la **variable auxiliaire** qui résout l'impossibilité d'indexer un tableau hôte par une
variable Z3 (`pool[platid[m][p]]` est interdit : l'index doit être un entier de la méta-théorie).
Total : **30 variables `Int`**.

In [5]:
// Variables de decision : 30 Int (PlatId 6 + Comp 18 + MenuNut 6).
var platid = Enumerable.Range(0, NB_MENUS)
    .Select(m => Enumerable.Range(0, NB_PLATS)
        .Select(p => ctx.MkIntConst($"pid_{m}_{p}")).ToArray()).ToArray();
var comp = Enumerable.Range(0, NB_MENUS * NB_PLATS)
    .Select(sl => Enumerable.Range(0, K)
        .Select(k => ctx.MkIntConst($"comp_{sl}_{k}")).ToArray()).ToArray();
var menunut = Enumerable.Range(0, NB_MENUS)
    .Select(m => Enumerable.Range(0, K)
        .Select(k => ctx.MkIntConst($"nut_{m}_{k}")).ToArray()).ToArray();
Console.WriteLine($"{NB_MENUS * NB_PLATS} PlatId + {NB_MENUS * NB_PLATS * K} Comp + {NB_MENUS * K} MenuNut = "
    + $"{NB_MENUS * NB_PLATS + NB_MENUS * NB_PLATS * K + NB_MENUS * K} variables Int.");

6 PlatId + 18 Comp + 6 MenuNut = 30 variables Int.


### Lecture du résultat — trente variables, six décisions

La décomposition compte 6 `platid` + 18 `comp` + 6 `menunut` = 30, mais toutes ne se valent
pas : seuls les 6 `platid` sont de **vraies décisions** (un index de candidat par slot). Les
18 `comp` sont déterminées par le `platid` qu'elles accompagnent, les 6 `menunut` par leurs
sommes. L'espace de choix effectif est donc bien plus petit que 30 dimensions — mais
l'encodage a besoin des 30 pour rester linéaire, sans fonction d'indexation hôte par une
variable Z3.

### 2.3 Le patient et ses restrictions (bornes par MENU)

Fidèle au C# `08` : un seul patient, **inline** comme quatre bornes entières, **par menu** (la somme
des 3 plats du menu doit respecter chaque borne). Bornes rescalées kcal→kJ (×4,184) au moment du
rewiring sur le cache Ciqual (kJ/100 g).

> **ATTENTION millésime (#8901)** : ces bornes ont été *réglées* contre un cache C# aujourd'hui
> désuet (`08` tourne sur 121 Ko / 736 recettes ; `07`+`09` s'accordent sur 270 Ko / 2387). Ici on
> consomme le **cache Python 16b régénéré frais** — les valeurs du menu résolu **différeront** des
> sorties C# d'origine (qu'on ne reproduit pas). On vérifie que le patient reste **SAT** sur ce
> pool (sinon = finding à reporter, pas de retouche silencieuse des bornes).

In [6]:
// Restrictions patient (fidele a Restriction{Min,Max}, -1 = pas de borne), par MENU.
const int energieMin = 3347;    // ~800 kcal cumules minimum par menu
const int energieMax = 10878;   // ~2600 kcal cumules maximum par menu
const int protMin = 30;         // proteines minimum par menu (g)
const int lipMax = 600;         // lipides maximum par menu (g)
Console.WriteLine($"Patient : energie in [{energieMin}, {energieMax}] kJ, proteines >= {protMin} g, "
    + $"lipides <= {lipMax} g (par menu)");

Patient : energie in [3347, 10878] kJ, proteines >= 30 g, lipides <= 600 g (par menu)


### Lecture du résultat — des bornes larges en apparence, qui mordent sur le pool réel

La fenêtre énergie [3347, 10878] kJ par menu semble accommodante (un rapport de 1 à 3,2), et
les bornes protéines/lipides n'ont l'air que de la prudence diététique. Rapportées au pool
curé, elles ne sont pas bénignes : le plat le plus énergétique du pool (15054 kJ) dépasse **à
lui seul** le plafond d'un menu entier — la contrainte élimine des combinaisons dès le premier
choix, avant même de composer.

Ces bornes sont celles du C# `08`, rescalées kcal vers kJ (× 4,184) au rewiring Ciqual — la mise
en garde du millésime s'applique : la vraie question n'est pas leur valeur, c'est si le patient
reste SAT sur ce cache Python frais. C'est exactement ce que la cellule suivante mesure.

### 2.4 Les cinq familles de contraintes

1. **Bornes d'ordre** : `platid[m][p]` dans la fenêtre des 5 candidats de son créneau.
2. **Variété** : `MkDistinct` global sur les 6 slots (pas deux fois le même plat).
3. **Linking composition** : `Or(platid[m][p] != cand, comp[slot][k] == teneur(cand, k))` — l'idiome
   **index + disjonction** (90 assertions) qui relie l'index choisi à sa composition. C'est l'idiome
   que le binding `Z3.Linq` produisait à votre place via `CollectionHandling.Array` ; ici il est
   écrit à la main, assertion par assertion.
4. **Somme par menu** : `menunut[m][k] == Σ comp[slot][k]` sur les 3 créneaux.
5. **Restrictions patient** : `menunut[m][·]` dans les bornes (énergie, protéines, lipides).

In [7]:
// THE MODEL : 5 familles de contraintes + Check (pure satisfiabilite, pas d'objectif).
var swModel = Stopwatch.StartNew();
var s = ctx.MkSolver();
// (1) bornes d'ordre
for (int m = 0; m < NB_MENUS; m++)
    for (int p = 0; p < NB_PLATS; p++) {
        s.Assert(ctx.MkGe(platid[m][p], ctx.MkInt(p * CAND)));
        s.Assert(ctx.MkLe(platid[m][p], ctx.MkInt(p * CAND + CAND - 1)));
    }
// (2) variete : Distinct global sur les 6 slots
s.Assert(ctx.MkDistinct(platid.SelectMany(r => r).Cast<Expr>().ToArray()));
// (3) linking composition : PlatId != cand || Comp[slot][k] == teneur(cand, k)
for (int m = 0; m < NB_MENUS; m++)
    for (int p = 0; p < NB_PLATS; p++) {
        int sl = Slot(m, p);
        for (int cand = p * CAND; cand < p * CAND + CAND; cand++)
            for (int k = 0; k < K; k++)
                s.Assert(ctx.MkOr(ctx.MkNot(ctx.MkEq(platid[m][p], ctx.MkInt(cand))),
                                  ctx.MkEq(comp[sl][k], ctx.MkInt(Teneur(cand, CONST[k])))));
    }
// (4) somme par menu
for (int m = 0; m < NB_MENUS; m++)
    for (int k = 0; k < K; k++)
        s.Assert(ctx.MkEq(menunut[m][k],
                          ctx.MkAdd(comp[Slot(m, 0)][k], comp[Slot(m, 1)][k], comp[Slot(m, 2)][k])));
// (5) restrictions patient (par menu)
for (int m = 0; m < NB_MENUS; m++) {
    s.Assert(ctx.MkGe(menunut[m][0], ctx.MkInt(energieMin)));
    s.Assert(ctx.MkLe(menunut[m][0], ctx.MkInt(energieMax)));
    s.Assert(ctx.MkGe(menunut[m][1], ctx.MkInt(protMin)));
    s.Assert(ctx.MkLe(menunut[m][2], ctx.MkInt(lipMax)));
}

bool satCap = s.Check() == Status.SATISFIABLE;
Model mdl = null;
if (satCap) mdl = s.Model;
swModel.Stop();
Console.WriteLine($"Modele resolu en {swModel.Elapsed.TotalMilliseconds:F0} ms");
if (!satCap) {
    Console.WriteLine("UNSAT : aucun plan ne satisfait les restrictions du patient sur cette pool (finding #8901).");
} else {
    for (int m = 0; m < NB_MENUS; m++) {
        long e = Val(mdl, menunut[m][0]), pr = Val(mdl, menunut[m][1]), li = Val(mdl, menunut[m][2]);
        Console.WriteLine($"== Menu {m + 1} ==  energie={e} kJ | proteines={pr} g | lipides={li} g");
        for (int p = 0; p < NB_PLATS; p++) {
            int v = (int)Val(mdl, platid[m][p]);
            Console.WriteLine($"   creneau {p + 1} : {pool[v].Nom.Trim(),-30} "
                + $"(kJ={Teneur(v, cEnergie)}, prot={Teneur(v, cProt)}, lip={Teneur(v, cLip)})");
        }
    }
}

Modele resolu en 18 ms


== Menu 1 ==  energie=8848 kJ | proteines=45 g | lipides=168 g


   creneau 1 : 4 B's Restaurant Tomato Soup   (kJ=3409, prot=16, lip=63)


   creneau 2 : (Sort-Of) Sweet and Sour Chicken (Also Good (kJ=1018, prot=3, lip=0)


   creneau 3 : ( From Bread Mix ) Big Soft Pretzels (kJ=4421, prot=26, lip=105)


== Menu 2 ==  energie=10571 kJ | proteines=111 g | lipides=119 g


   creneau 1 : 'ncapriata Di Fave (Fava Bean Puree with Gre (kJ=5922, prot=58, lip=73)


   creneau 2 : 10 Minute Szechuan Chicken     (kJ=1178, prot=11, lip=15)


   creneau 3 : 1-Pot: Creamy Chicken Noodle Casserole (kJ=3471, prot=42, lip=31)


### Lecture du résultat — deux menus SAT, à des distances très différentes des bornes

Résolution en quelques dizaines de millisecondes. Le Menu 1 (soupe 3409 + sweet-sour 1018 +
pretzels 4421 kJ) totalise 8848 kJ, 45 g de protéines, 168 g de lipides ; le Menu 2 (purée de
fèves 5922 + szechuan 1178 + casserole 3471 kJ) totalise 10571 kJ, 111 g, 119 g. Les deux
respectent les trois bornes, et les six plats sont deux à deux distincts — la contrainte de
variété se lit dans la composition même des menus.

La lecture la plus riche est la **distance aux bornes** : le Menu 2 s'arrête à 307 kJ du
plafond d'énergie (2,8 % sous 10878) et multiplie le plancher protéines par 3,7 (111 g contre
30), là où le Menu 1 reste nettement en retrait du plafond (8848 kJ). Le solveur n'optimise
rien — il rend le premier témoin SAT — et déjà les deux témoins occupent des positions très
différentes du polytope. La mise en garde du §2.3 se referme : patient SAT sur le cache frais,
aucun finding à reporter, les bornes du millésime n'ont pas eu à être retouchées.

### 2.5 Vérification hors-Z3 (croire vs prouver)

Un solveur **garantit** les contraintes par construction — mais on **vérifie** indépendamment
(re-calcul hors Z3). Deux prongs : (a) les sommes recomputées depuis la pool satisfont les bornes
patient ; (b) ces sommes **égalent** les `menunut` du solveur — ce qui valide la **couche de
linking** elle-même, pas seulement les bornes.

In [8]:
// Re-verification hors Z3 : re-somme depuis la pool, compare aux MenuNut du solveur.
if (!satCap) {
    Console.WriteLine("Pas de solution a verifier (UNSAT).");
} else {
    bool ok = true;
    for (int m = 0; m < NB_MENUS; m++) {
        int e = 0, pr = 0, li = 0;
        for (int p = 0; p < NB_PLATS; p++) {
            int v = (int)Val(mdl, platid[m][p]);
            e += Teneur(v, cEnergie); pr += Teneur(v, cProt); li += Teneur(v, cLip);
        }
        bool eOk = energieMin <= e && e <= energieMax, pOk = pr >= protMin, lOk = li <= lipMax;
        long Me = Val(mdl, menunut[m][0]), Mpr = Val(mdl, menunut[m][1]), Mli = Val(mdl, menunut[m][2]);
        bool match = e == Me && pr == Mpr && li == Mli;
        ok = ok && eOk && pOk && lOk && match;
        Console.WriteLine($"Menu {m + 1} : recalc energie={e}({(eOk ? "OK" : "HORS")}), "
            + $"prot={pr}({(pOk ? "OK" : "HORS")}), lip={li}({(lOk ? "OK" : "HORS")}) ; "
            + $"coherent avec MenuNut={(match ? "oui" : "NON")}");
    }
    Console.WriteLine(ok ? "\nVERIFIE : toutes les restrictions patient sont respectees."
                         : "\nINCOHERENCE detectee (a investiguer).");
}

Menu 1 : recalc energie=8848(OK), prot=45(OK), lip=168(OK) ; coherent avec MenuNut=oui


Menu 2 : recalc energie=10571(OK), prot=111(OK), lip=119(OK) ; coherent avec MenuNut=oui



VERIFIE : toutes les restrictions patient sont respectees.


### Lecture du résultat — six recalculs, zéro écart

Chacun des six chiffres recomptés hors Z3 (2 menus × 3 constituants) reproduit exactement la
valeur du solveur, et les bornes sont repassées une à une : le menu n'est pas seulement SAT au
sens de Z3, il est **vérifié** en arithmétique ordinaire. Le détail qui compte est le second
prong : l'égalité avec `menunut` ne revalide pas les bornes (elles viennent d'être repassées),
elle valide la **couche de linking** — les 90 disjonctions qui relient chaque `platid` à la
composition de son candidat. Si une seule disjonction avait relié un mauvais index, un écart
apparaîtrait ici entre la somme recomptée et la somme du solveur.

C'est la différence entre croire et prouver : le témoin de SAT vient du solveur ; la preuve que
le témoin est ce qu'il prétend être vient du recalcul.

## 3. Exercices

Quatre prolongements (stubs : ils s'exécutent tels quels, n'échouent pas, à compléter).

**Exercice 1** — Diversité d'entrée sur jours consécutifs : forcer que le créneau 1 diffère
entre `Menu 1` et `Menu 2` (déjà couvert par le `Distinct` global, mais l'exprimer comme une
contrainte dédiée `Or(platid[0][0] != platid[1][0], ...)` et mesurer l'impact).

In [9]:
// Exercice 1 (stub) : diversite entree jours consecutifs.
// TODO etudiant : ajouter s.Assert(ctx.MkOr(...)) forcant creneau 1 different entre menus,
//                 re-solve, comparer au nombre de plans deja obtenu.
int ExerciceDiversiteEntree()
{
    // TODO etudiant : implementez la contrainte dediee et retournez le nombre de plans
    return -1;  // TODO etudiant : remplacer par le resultat mesure
}
Console.WriteLine($"[Ex.1] Stub : a completer (diversite entree jours consecutifs) -> {ExerciceDiversiteEntree()}");

[Ex.1] Stub : a completer (diversite entree jours consecutifs) -> -1


**Exercice 2** — Seuil d'énergie strict → UNSAT : resserrer `energieMax` par dichotomie
jusqu'à rendre le modèle insatisfiable. Quel créneau porte l'essentiel de l'énergie kJ ?

In [10]:
// Exercice 2 (stub) : seuil UNSAT par dichotomie sur energieMax.
// TODO etudiant : boucle resserrant energieMax, re-solve a chaque pas, retourner le seuil ou ca casse.
int ExerciceUnsatThreshold()
{
    // TODO etudiant : dichotomie sur energieMax (re-construire le solveur a chaque essai)
    return -1;  // TODO etudiant : remplacer par le seuil mesure
}
Console.WriteLine($"[Ex.2] Stub : a completer (seuil UNSAT energie) -> {ExerciceUnsatThreshold()}");

[Ex.2] Stub : a completer (seuil UNSAT energie) -> -1


**Exercice 3** — Passage à 3 menus : adapter les dimensions des matrices (`platid` 3×3,
`comp` 9×3, `menunut` 3×3) et re-solve. Note : les formes sont hardcodées à `NB_MENUS = 2` dans la
déclaration des variables — les paramétrer est le cœur de l'exercice.

In [11]:
// Exercice 3 (stub) : passer a 3 menus (adapter les formes de matrices).
// TODO etudiant : parametrer NB_MENUS=3, redeclarer les matrices, re-solve, retourner le temps.
double ExerciceTroisMenus()
{
    // TODO etudiant : NB_MENUS = 3, redeclarer platid/comp/menunut, re-solve
    return -1.0;  // TODO etudiant : remplacer par le temps mesure (ms)
}
Console.WriteLine($"[Ex.3] Stub : a completer (3 menus) -> {ExerciceTroisMenus()}");

[Ex.3] Stub : a completer (3 menus) -> -1


**Exercice 4** — Ajouter le sel (`Sel chlorure de sodium`, cache idx 4) comme 4e constituant
contraint (borne max `selMax`), et re-solve.

In [12]:
// Exercice 4 (stub) : ajouter le constituant Sel (cache idx 4).
// TODO etudiant : ajouter l'index a CONST, definir selMax, redeclarer comp/menunut sur K=4, re-solve.
int ExerciceConstituantSel()
{
    int cSel = Array.FindIndex(constituants.ToArray(),
        n => n.ToLowerInvariant().Contains("sel chlorure"));
    Console.WriteLine($"  indice trouve dans le cache : constituant[{cSel}]");
    // TODO etudiant : etendre CONST avec cSel, borner le sel, re-solve
    return -1;  // TODO etudiant : remplacer par la valeur mesuree une fois la contrainte posee
}
Console.WriteLine($"[Ex.4] Stub : a completer (constituant sel) -> {ExerciceConstituantSel()}");

  indice trouve dans le cache : constituant[4]


[Ex.4] Stub : a completer (constituant sel) -> -1


## Synthèse — le capstone patient, en C# natif

| Mouvement | Ce qu'il montre | Idiome Z3 |
|---|---|---|
| **M1 squelette** | semaine 7×5 sans nutrition | grille plate de constantes `Int` (même choix que z3-py ; le `08` d'origine passait par `CollectionHandling.Array`) |
| **M2 capstone** | patient SAT avec restrictions nutritionnelles | **index + linking disjonction** (`MkOr(pid ≠ cand, comp == val)`), écrit à la main hors `Z3.Linq` |

L'arc complet se lit comme un entonnoir : 1465 recettes brutes, 447 exploitables dans le cache,
15 candidats dans le pool curé — et, au bout, un modèle de 30 variables dont seules 6 sont de
vraies décisions, résolu en quelques dizaines de millisecondes. Le capstone patient est **le point
où la nutrition rencontre la combinatoire** : on ne peut pas forcer « 30 g de protéines minimum »
sur un jouet de 24 plats — il faut un corpus réel (le cache 16b) pour que la contrainte *morde*,
jusqu'au plat unique qui dépasse à lui seul le plafond d'énergie du menu. La **re-vérification
hors-Z3** clôt l'arc : six recalculs sans écart — on ne *croit* pas le solveur, on le *prouve*.

Ce que ce carnet ajoute à la jambe C# de la série : la série sœur `Z3-Linq2Z3` **déclarait** ce
capstone en `int[][]` et laissait la couche de binding le traduire ; ici le même problème est
posé en `Microsoft.Z3` nu, où chaque variable et chaque disjonction sont visibles — c'est la
contrepartie exacte du carnet Python, dans le langage de la série sœur.

**Prochain grain (G3)** : la convergence à l'échelle — comparer les encodages (index naïf vs
one-hot pseudo-Boolean `MkPBEq`/`MkPBLe`) quand `R` et `C` augmentent, là où le naïf explose
(c'est l'objet de [16d](Z3-16d-Meal-Planner-Convergence-Scale-Python.ipynb) côté Python, et de son
jumeau C# à venir).